# Lecture

## Problems of transformers

- Costs: GPT-3 train cost is estimated to be 10-50 billion USD
- Data: Llama 3 was trained on over 15T tokens, that's roughly 10 trillion words
- Complexity of self-attention
  $$
    \operatorname{Attention} = \operatorname{softmax}(QK^\top / \sqrt{d_k}) V
  $$

In BERT maximum sequence length is 512 tokens. Real-world sequences are well beyond 1k tokens:
- Summarization: 781 tokens in CNN / Daily Mail dataset
- Promoter Region Prediction from DNA: about 4k tokens in the dataset
- Bank Transactions: more than 1k tokens during one year

## Solution: separable attention

- Split matrix into blocks
- Calculate attention within blocks
- Join the attention from blocks

Problems:
- Not natural
- Added hyperparameters
- Losing cross-block context

## Efficient transformers (as of 2021)

<img src="efficient-transformers.png" width=800px/>

## Longformer

<table>
    <tr>
        <td>
            <img src="longformer-1.png" width=600px/>
            <br>
            <ul>
                <li>Full $O(n^2)$</li>
                <li>Sliding window $O(hn)$</li>
                <li>Global $O(gn)$, examples: [CLS], [SEP]</li>
                <li>Total $O((g + h)n)$</li>
            </ul>
        </td>
        <td><img src="longformer-2.png" width=400px/></td>
    </tr>
</table>

## BigBird

<table>
    <tr>
        <td>
            <ul>
                <li>Full $O(n^2)$</li>
                <li>Random $O(rn)$</li>
                <li>Sliding window $O(hn)$</li>
                <li>Global $O(gn)$</li>
                <li>Total $O((r + g + h)n)$</li>
            </ul>
            <ul>
                <li>Sparse-attention transformer is a universal approximator</li>
                <li>Sparse-attention transformer is Turing-complete</li>
                <li>For a single layer of full attention, an equivalent sparse-attention requires up to $n$ layers</li>
            </ul>
            Attention blocks can be reshaped in way to better utilize GPU
        </td>
        <td><img src="bigbird.png" width=300px/></td>
    </tr>
</table>

## Reformer

<table>
    <tr>
        <td>
            <ol>
                <li>Apply locality-sensitive hashing using random projections</li>
                <li>Apply random rotations to determine similar tokens, split into buckets</li>
            </ol>
            <img src="reformer-2.png" width=500px/>
        </td>
        <td><img src="reformer-1.png" width=400px/></td>
    </tr>
</table>

3. Sort by bucket, split uniformly into chunks
4. Only allow attention within a bucket and at most previous chunk

<table>
    <tr>
        <td>
            <img src="reformer-3.png" width=500px/>
        </td>
        <td>
            <img src="reformer-4.png" width=300px/>
        </td>
    </tr>
</table>

## Learnable patterns

<table>
    <tr>
        <td>
            <ul>
                <li>Fixed patterns -> Learned patterns (from data)</li>
                <li>Usually revolves around defining token relevance measure</li>
            </ul>
            <ul>
                <li>Hard to implement efficiently</li>
                <li>Slower</li>
            </ul>
        </td>
        <td><img src="learnable-patterns.png" width=300px/></td>
    </tr>
</table>

## Low rank approximations

<table>
    <tr>
        <td>
            <ul>
                <li>Assumne low-rank structure of self-attention matrix</li>
                <li>Usually revolves around decomposition</li>
            </ul>
            <ul>
                <li>Can't optimize directly the low-rank matrices</li>
            </ul>
        </td>
        <td><img src="lowrank.png" width=300px/></td>
    </tr>
</table>

## Linformer

<table>
    <tr>
        <td>
            <ul>
                <li>Perform SVD of each self-attention matrix</li>
                <li>Project keys and values onto smaller dimension</li>
            </ul>
        </td>
        <td><img src="linformer.png" width=500px/></td>
    </tr>
</table>

## Faster transformers

<img src="faster-transformers.png" width=500px/>

## Performer: Fast Attention Via positive Orthogonal Random features

<table>
    <tr>
        <td>
            <ul>
                <li>
                    Full attention: $O(L^2d)$ time, $O(L^2 + Ld)$ space complexity
                    <br>
                    $$
                    \operatorname{Att}(\mathbf{Q}, \mathbf{K}, \mathbf{V})=\mathbf{D}^{-1} \mathbf{A V}, \quad \mathbf{A}=\exp \left(\mathbf{Q} \mathbf{K}^{\top} / \sqrt{d}\right), \quad \mathbf{D}=\operatorname{diag}\left(\mathbf{A} \mathbf{1}_L\right) .
                    $$
                </li>
                <li>
                    Low-rank attention: $O(Lrd)$ time, $O(Lr + Ld + rd)$ space complexity
                    <br>
                    $$
                    \widehat{\operatorname{Att}}(\mathbf{Q}, \mathbf{K}, \mathbf{V})=\widehat{\mathbf{D}}^{-1}\left(\mathbf{Q}^{\prime}\left(\left(\mathbf{K}^{\prime}\right)^{\top} \mathbf{V}\right)\right), \quad \widehat{\mathbf{D}}=\operatorname{diag}\left(\mathbf{Q}^{\prime}\left(\left(\mathbf{K}^{\prime}\right)^{\top} \mathbf{1}_L\right)\right) .
                    $$
                    where $\mathbf{Q}^{\prime}, \mathbf{K}^{\prime} \in \mathbb{R}^{L \times r}$ are matrices with rows $\phi(q_i^\top)^\top$ and $\phi(k_i^\top)^\top$ respectively.
                </li>
            </ul>
        </td>
        <td><img src="performer-1.png" width=500px/></td>
    </tr>
</table>

## Performer: FAVOR+

- Feature vector
  $$
  \phi(\mathbf{x})=\frac{h(\mathbf{x})}{\sqrt{m}}\left(f_1\left(\omega_1^{\top} \mathbf{x}\right), \ldots, f_1\left(\omega_m^{\top} \mathbf{x}\right), \ldots, f_l\left(\omega_1^{\top} \mathbf{x}\right), \ldots, f_l\left(\omega_m^{\top} \mathbf{x}\right)\right)
  $$
- Softmax kernel:
  $$
  \operatorname{SM}(\mathbf{x}, \mathbf{y})=\exp \left(\frac{\|\mathbf{x}\|^2}{2}\right) \mathrm{K}_{\text {gauss }}(\mathbf{x}, \mathbf{y}) \exp \left(\frac{\|\mathbf{y}\|^2}{2}\right)
  $$
  $$
  h(x) = \exp \left(\frac{\|\mathbf{x}\|^2}{2}\right), \; l=2, \; f_1 = \sin, \; f_2 = \cos
  $$
- Softmax kernel:
  $$
  \operatorname{SM}(\mathbf{x}, \mathbf{y})=\mathbb{E}_{\omega \sim \mathcal{N}\left(0, \mathbf{I}_d\right)}\left[\exp \left(\omega^{\top} \mathbf{x}-\frac{\|\mathbf{x}\|^2}{2}\right) \exp \left(\omega^{\top} \mathbf{y}-\frac{\|\mathbf{y}\|^2}{2}\right)\right]=\Lambda \mathbb{E}_{\omega \sim \mathcal{N}\left(0, \mathbf{I}_d\right)} \cosh \left(\omega^{\top} \mathbf{z}\right)
  $$
  $$
  h(x) = \exp \left(\frac{\|\mathbf{x}\|^2}{2}\right), \; l=1, \; f_1 = \exp
  $$
  $$
    h(x) = \frac{1}{\sqrt{2}}\exp \left(\frac{\|\mathbf{x}\|^2}{2}\right), \; l=2, \; f_1(u) = \exp(u), \; f_2(u) = \exp(-u)
  $$

## FlashAttention

<table>
    <tr>
        <td><img src="flash-attention.png" width=600px/></td>
        <td><img src="flash-attention-2.png" width=400px/></td>
    </tr>
</table>

## Transformers for Time Series

<img src="transformers-time-series.png" width=800px/></td>

## Informer

<table>
    <tr>
        <td>
            Two major improvements over Transformer:
            <ul>
                <li>Sparse Attention with $O(T \log T)$ time and space complexity</li>
                <li>Shrinking the input size between layers when stacking\</li>
            </ul>
        </td>
        <td><img src="informer-1.png" width=400px/></td>
    </tr>
</table>

## Informer: ProbSparse Attention

<table>
    <tr>
        <td>
            <ul>
                <li>
                    $$
                        \operatorname{ProbSparseAttention} = \operatorname{softmax}(Q_{reduce}K^\top / \sqrt{d_k}) V
                    $$
                </li>
                <li>Fill $Q_{reduce}$ with active queries</li>
                <li>Determine active queries by computing the KL-divergence over sampled keys</li>
                <li>
                    $$
                    M\left(\mathbf{q}_i, \mathbf{K}\right)=\max _j\left\{\frac{\mathbf{q}_i \mathbf{k}_j^{\top}}{\sqrt{d}}\right\}-\frac{1}{L_K} \sum_{j=1}^{L_K} \frac{\mathbf{q}_i \mathbf{k}_j^{\top}}{\sqrt{d}}
                    $$
                </li>
            </ul>
        </td>
        <td><img src="informer-3.png" width=400px/></td>
    </tr>
</table>

## Informer: Distilling

<table>
    <tr>
        <td>$$X_{n+1} = \operatorname{MaxPool}\left(\operatorname{ELU}\left(\operatorname{Conv1d}(X_n)\right)\right)$$</td>
        <td><img src="informer-2.png" width=600px/></td>
    </tr>
</table>

## Autoformer

<table>
    <tr>
        <td>
            Two major improvements over Transformer:
            <ul>
                <li>Decomposition layer</li>
                <li>Autocorrelation as attention</li>
            </ul>
        </td>
        <td><img src="autoformer-1.png" width=600px/></td>
    </tr>
</table>

## Autoformer: decomposition layer

<table>
    <tr>
        <td>
            $$
                \begin{aligned}
                    X_{\text trend} & = \operatorname{AvgPool} \left( \operatorname{Padding} \left( X \right) \right), \\
                    X_{\text seasonal} & = X - X_{\text trend}
                \end{aligned}
            $$
        </td>
        <td><img src="autoformer-3.png" width=600px/></td>
    </tr>
</table>

## Autoformer: autocorrelation as attention

<table>
    <tr>
        <td>
            $$
            \begin{aligned}
            \tau_1, \cdots, \tau_k & =\underset{\tau \in\{1, \cdots, L\}}{\arg \operatorname{Topk}}\left(\mathcal{R}_{\mathcal{Q}, \mathcal{K}}(\tau)\right) \\
            \widehat{\mathcal{R}}_{\mathcal{Q}, \mathcal{K}}\left(\tau_1\right), \cdots, \widehat{\mathcal{R}}_{\mathcal{Q}, \mathcal{K}}\left(\tau_k\right) & =\operatorname{SoftMax}\left(\mathcal{R}_{\mathcal{Q}, \mathcal{K}}\left(\tau_1\right), \cdots, \mathcal{R}_{\mathcal{Q}, \mathcal{K}}\left(\tau_k\right)\right) \\
            \text { Auto-Correlation }(\mathcal{Q}, \mathcal{K}, \mathcal{V}) & =\sum_{i=1}^k \operatorname{Roll}\left(\mathcal{V}, \tau_i\right) \widehat{\mathcal{R}}_{\mathcal{Q}, \mathcal{K}}\left(\tau_i\right),
            \end{aligned}
            $$
            </td>
        <td><img src="autoformer-2.png" width=600px/></td>
    </tr>
</table>

# Webinar